# ReAct, Planejamento e Reflexão

Um agente pode organizar o próprio trabalho de três formas.

No **ReAct**, ele decide o próximo passo a cada rodada, a partir do resultado do passo anterior. No **planejamento**, ele escreve a sequência inteira antes de executar qualquer coisa. Na **reflexão**, ele produz uma resposta, avalia essa resposta segundo critérios e a reescreve enquanto a avaliação apontar problemas.

Cada seção tem a sua tarefa, as suas ferramentas e nenhum estado em comum com as outras.

In [ ]:
# No Google Colab, descomente e rode uma vez.
# !pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai"

import json
import urllib.request
from pathlib import Path

from pydantic import BaseModel, Field

from agentkit import LLMAPI, Agent, run_python, tool

O modelo desta aula escolhe ferramentas, escreve planos e avalia textos.

In [ ]:
MODEL_NAME = "gpt-4.1-nano"
llm = LLMAPI(MODEL_NAME, temperature=0.0, max_tokens=800)
print(llm.model)

In [ ]:
# Para rodar com um modelo local, use esta célula no lugar da anterior.
# O restante do notebook não muda.

# import torch
# from agentkit import LLM

# llm = LLM("Qwen/Qwen2.5-3B-Instruct", temperature=0.0, max_tokens=800)
# print(llm.model)

In [ ]:
WORKSPACE = Path("workspace")
WORKSPACE.mkdir(exist_ok=True)

## ReAct

O nome vem de *reasoning and acting*. O agente alterna raciocínio e ação: escolhe uma ferramenta, recebe o resultado dela como observação, e usa essa observação para escolher a ferramenta seguinte. O ciclo termina quando ele responde sem pedir mais nada.

A tarefa desta seção é sequencial por construção. A conversão depende da cotação que a API devolve, e o arquivo depende do valor convertido. Cada ação só pode ser escolhida depois que a anterior entregou o resultado.

In [ ]:
@tool
def exchange_rate() -> str:
    """Devolve quantos reais vale um dólar americano agora."""
    url = "https://economia.awesomeapi.com.br/json/last/USD-BRL"
    request = urllib.request.Request(url, headers={"User-Agent": "agentkit"})
    with urllib.request.urlopen(request, timeout=15) as response:
        return json.load(response)["USDBRL"]["bid"]


@tool
def calculator(expression: str) -> str:
    """Calcula uma expressão aritmética, como 12 * (3 + 4)."""
    return str(eval(expression, {"__builtins__": {}}, {}))


@tool
def write_file(name: str, content: str) -> str:
    """Escreve um arquivo de texto na pasta workspace."""
    (WORKSPACE / name).write_text(content, encoding="utf-8")
    return f"{name} gravado"

O `Agent` do `agentkit` é esse laço, construído no notebook de ferramentas.

In [ ]:
TASK = """Consulte a cotação atual do dólar em reais.
Calcule quanto US$ 1847,50 valem em reais.
Salve o resultado em cotacao.txt."""

messages = Agent(llm, [exchange_rate, calculator, write_file], max_steps=5).run(TASK)

for message in messages:
    print(message["role"], ":", message.get("content") or message["tool_calls"])

In [ ]:
print((WORKSPACE / "cotacao.txt").read_text())

O histórico traz o ciclo inteiro: três pedidos de ferramenta, três observações de volta e a resposta final.

O argumento da `calculator` é a cotação que a `exchange_rate` devolveu, e o texto gravado carrega o valor que a `calculator` devolveu. Nenhum desses números aparece no pedido original nem no código das ferramentas: eles entram no laço como observação e saem dele como argumento da ação seguinte.

### Exercício 1

Monte um dicionário com alguns usuários, cada um com id, nome e créditos, e crie duas ferramentas sobre ele: uma que procura um usuário pelo nome e devolve os dados, e outra que soma créditos a um usuário identificado pelo id. Peça ao agente que acrescente 10 créditos à Ana.

Responda quantas ações o agente executou e de onde saiu o id que ele passou para a segunda ferramenta.

In [ ]:
# Seu código aqui

## Planejamento

Planejar separa a decisão da execução. O agente escreve primeiro a sequência de passos que pretende seguir, e essa sequência fica disponível como dado antes de qualquer ação acontecer. Um plano pode ser lido, contado e alterado enquanto ainda não custou nada.

A ferramenta desta seção é o `run_python`, que executa código e devolve o que ele imprimir ou o valor da última expressão. As variáveis criadas em um passo continuam disponíveis no passo seguinte.

In [ ]:
# A entrada é código, e a saída é texto: "ok" quando nada é produzido, o valor da
# última expressão, o que for impresso, ou a mensagem do erro.
print(run_python("total = 2 + 3"))
print(run_python("total * 10"))
print(run_python('print(f"o total é {total}")'))
print(run_python("total / 0"))

O plano é uma saída estruturada. Cada passo tem a instrução que o agente vai receber e o resultado que se espera dela, e o esquema garante que a resposta chega nesse formato. O papel do planejador fica na mensagem de sistema, e a tarefa vai na mensagem do usuário.

In [ ]:
class Step(BaseModel):
    instruction: str
    expected_result: str


class Plan(BaseModel):
    steps: list[Step] = Field(min_length=2)


PLANNER_SYSTEM = """Você escreve planos para um agente que executa código Python, um passo por vez.
Cada passo tem uma instrução e o resultado que se espera dela.
Ainda não escreva código."""

TASK = """Compare sen(x) e cos(x) no intervalo [0, 2*pi] e mostre num gráfico em que ponto
a soma das duas funções é máxima. Salve o gráfico no arquivo "workspace/funcao.png"."""

plan = llm.generate_structured(
    [{"role": "system", "content": PLANNER_SYSTEM}, {"role": "user", "content": TASK}],
    Plan,
    max_tokens=700,
)

for number, step in enumerate(plan.steps, start=1):
    print(f"{number}. {step.instruction}")
    print(f"   espera: {step.expected_result}")

Nada foi executado ainda. A execução percorre os passos com um laço: cada passo entra como a próxima mensagem de uma conversa que continua, e o `Agent` resolve aquele passo com a ferramenta. A instrução vai acompanhada do resultado esperado, que delimita onde o passo termina.

In [ ]:
EXECUTOR_SYSTEM = "Você executa um passo por vez, rodando código Python."

messages = [{"role": "system", "content": EXECUTOR_SYSTEM}]

for step in plan.steps:
    messages = Agent(llm, [run_python], max_steps=4).run(
        messages
        + [{"role": "user", "content": f"{step.instruction}\nResultado esperado: {step.expected_result}"}]
    )
    print(step.instruction)
    print("  ", (messages[-1]["content"] or "").strip()[:160], "\n")

In [ ]:
from IPython.display import Image

Image(WORKSPACE / "funcao.png")

Cada passo do plano virou um trecho de código, na ordem planejada, e o arquivo só aparece no último passo.

O que liga um passo ao seguinte é o estado do interpretador: o `x` gerado no começo continua existindo quando o gráfico é montado, e a soma calculada no meio é usada para marcar o ponto de máximo.

A diferença para a seção anterior está no momento em que a sequência foi decidida. No ReAct, cada ação só podia ser escolhida depois de ver a observação da anterior. Aqui os passos já existiam antes da primeira execução, e o laço apenas os percorre.

### Exercício 2

Mude a tarefa para pedir também a derivada aproximada de sen(x) no mesmo gráfico, e compare o plano gerado com o anterior. Responda quantos passos entraram e se algum passo antigo mudou de conteúdo.

In [ ]:
# Seu código aqui

## Reflexão

Refletir é avaliar a própria resposta segundo critérios e reescrevê-la a partir do que a avaliação apontou. O ciclo se repete enquanto houver problema, e termina quando a avaliação aprova ou quando as rodadas acabam.

Aqui o texto é escrito sem que o autor conheça as regras. As regras pertencem ao avaliador, e chegam ao autor só como a lista do que ficou faltando.

In [ ]:
text = llm.invoke(
    [{"role": "user", "content": "Escreva um texto curto sobre inteligência artificial."}],
    max_tokens=400,
).strip()
print(text)

Cada regra vira um campo booleano do esquema `Review`, e o avaliador responde uma pergunta por regra. A nota é a soma das regras cumpridas, e a lista de problemas é montada com as regras que voltaram falsas.

In [ ]:
class Review(BaseModel):
    gives_an_agriculture_example: bool
    mentions_the_energy_cost: bool
    gives_an_advantage: bool


RULES = {
    "gives_an_agriculture_example": "dar um exemplo de IA usada na agricultura",
    "mentions_the_energy_cost": "mencionar o custo de energia para treinar modelos de IA",
    "gives_an_advantage": "apresentar uma vantagem da IA",
}

REVIEW_PROMPT = """Responda uma pergunta para cada regra sobre o texto abaixo.

O texto dá um exemplo de IA usada na agricultura?
O texto menciona o custo de energia para treinar modelos de IA?
O texto apresenta uma vantagem da IA?

Texto:
{text}"""


def reflect(text: str) -> tuple[int, list[str]]:
    """Devolve quantas regras o texto cumpre e a lista do que falta."""
    review = llm.generate_structured(
        [{"role": "user", "content": REVIEW_PROMPT.format(text=text)}], Review, max_tokens=200
    )
    missing = [rule for field, rule in RULES.items() if not getattr(review, field)]
    return len(RULES) - len(missing), missing

In [ ]:
REWRITE_PROMPT = """Reescreva o texto para que ele também faça o seguinte:

{problems}

Texto:
{text}

Responda apenas com o novo texto."""

for number in range(5):
    score, problems = reflect(text)
    print(f"VERSÃO {number} | nota {score}/{len(RULES)}")
    for problem in problems:
        print("  falta:", problem)
    if not problems:
        break
    text = llm.invoke(
        [{"role": "user", "content": REWRITE_PROMPT.format(
            text=text, problems="\n".join(f"- {problem}" for problem in problems))}],
        max_tokens=500,
    ).strip()

print()
print(text)

A nota subiu de 1 para 3 e o laço parou na segunda versão.

As duas saídas do avaliador têm papéis distintos no laço. A nota decide se ele continua, e a lista de problemas alimenta a reescrita, porque nomeia o que precisa entrar no texto. O autor nunca viu as regras: conheceu cada uma pelo que o avaliador apontou como faltando.

### Exercício 3

Faça duas mudanças no laço, uma de cada vez, e responda o que acontece em cada uma.

Primeiro, esvazie a lista de problemas que vai para a reescrita, mantendo a nota como está. Rode as rodadas todas e diga o que acontece com a nota e com o tamanho do texto.

Depois volte a lista ao normal e acrescente uma quarta regra: citar uma fonte com autor e ano. Rode de novo e diga em que rodada o laço parou, o que passou a aparecer no texto, e o que o avaliador teria que fazer para conferir aquilo de verdade.

In [ ]:
# Seu código aqui

## Quando usar cada uma

| técnica | a tarefa pede |
| --- | --- |
| ReAct | o próximo passo depende do resultado do anterior |
| Planejamento | a sequência pode ser decidida antes de executar |
| Reflexão | o resultado é avaliado por critérios, sem resposta única |

As três se combinam, e é daí que a Unidade II parte: um agente que planeja, executa cada passo com ReAct e reflete sobre o resultado antes de entregar.